# Stage 4 — Tokenization

We'll walk through three tokenizer flavors, smallest → most useful:

1. **Whitespace** — naive baseline, shows the OOV problem.
2. **From-scratch BPE** — to *see* how Byte-Pair Encoding works (merge pairs of bytes by frequency).
3. **`tiktoken` GPT-2 BPE** — production-grade; what we'll use for the dataloader.

We also build a `GPTDatasetV1` that emits `(input, target)` pairs via a sliding window — exactly what the LLM trains on.

In [1]:
from pathlib import Path
train = Path('data/processed/train.txt').read_text(encoding='utf-8')
val   = Path('data/processed/val.txt').read_text(encoding='utf-8')
print(f'train: {len(train):,} chars   val: {len(val):,} chars')
print(train[:300])

train: 1,531,067 chars   val: 170,119 chars
Adi Parva

Chapter One
Maharaja Shantanu Marries the Celestial Ganga

According to the historical records of this earth, there once lived a King named Maharaja Shantanu, the son of Pratipa, who took his birth in the solar dynasty and was considered naradeva, the manifest representative of the Suprem


## 1) Naive whitespace tokenizer

In [2]:
import re

class WhitespaceTokenizer:
    def __init__(self, text: str):
        tokens = re.findall(r"[\w']+|[.,!?;:\"]", text)
        vocab = sorted(set(tokens))
        vocab.extend(['<|unk|>', '<|endoftext|>'])
        self.stoi = {t: i for i, t in enumerate(vocab)}
        self.itos = {i: t for t, i in self.stoi.items()}

    def encode(self, text):
        toks = re.findall(r"[\w']+|[.,!?;:\"]", text)
        unk = self.stoi['<|unk|>']
        return [self.stoi.get(t, unk) for t in toks]

    def decode(self, ids):
        return ' '.join(self.itos[i] for i in ids)

ws = WhitespaceTokenizer(train)
print('vocab size:', len(ws.stoi))
sample = 'Arjuna spoke to Krishna on the battlefield of Kurukshetra'
ids = ws.encode(sample)
print('ids   :', ids[:15])
print('decoded:', ws.decode(ids))

vocab size: 11222
ids   : [234, 9707, 10303, 1194, 7757, 10198, 3165, 7719, 1235]
decoded: Arjuna spoke to Krishna on the battlefield of Kurukshetra


## 2) BPE from scratch (educational)

Algorithm:
1. Start with the byte vocabulary (256 symbols).
2. Count adjacent pair frequencies across the corpus.
3. Merge the most frequent pair into a new symbol.
4. Repeat until you hit a target vocab size.

We use a tiny `num_merges` on a slice of the corpus so it finishes in seconds.

In [3]:
from collections import Counter

def get_pair_stats(ids):
    return Counter(zip(ids, ids[1:]))

def merge(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

def train_bpe(text: str, num_merges: int = 200):
    ids = list(text.encode('utf-8'))
    merges, next_id = {}, 256
    for step in range(num_merges):
        stats = get_pair_stats(ids)
        if not stats:
            break
        pair, _ = stats.most_common(1)[0]
        ids = merge(ids, pair, next_id)
        merges[pair] = next_id
        next_id += 1
    return merges

def encode_bpe(text: str, merges: dict):
    ids = list(text.encode('utf-8'))
    while len(ids) >= 2:
        pairs = get_pair_stats(ids)
        # apply the earliest-learned merge present in the text
        pair = min(pairs, key=lambda p: merges.get(p, float('inf')))
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids

# Train on a slice so the demo is fast.
slice_ = train[:100_000]
merges = train_bpe(slice_, num_merges=300)
print(f'learned {len(merges)} merges → vocab size {256 + len(merges)}')

demo = 'Arjuna spoke to Krishna'
raw  = list(demo.encode('utf-8'))
enc  = encode_bpe(demo, merges)
print(f'raw bytes ({len(raw)}): {raw}')
print(f'BPE ids   ({len(enc)}): {enc}')
print(f'compression: {len(raw) / len(enc):.2f}x')

learned 300 merges → vocab size 556
raw bytes (23): [65, 114, 106, 117, 110, 97, 32, 115, 112, 111, 107, 101, 32, 116, 111, 32, 75, 114, 105, 115, 104, 110, 97]
BPE ids   (11): [65, 114, 106, 315, 270, 418, 111, 429, 286, 507, 97]
compression: 2.09x


## 3) `tiktoken` GPT-2 BPE (what we'll actually use)

In [4]:
import tiktoken
tok = tiktoken.get_encoding('gpt2')
print('GPT-2 vocab size:', tok.n_vocab)

sample = 'Arjuna spoke to Krishna on the battlefield of Kurukshetra <|endoftext|>'
ids = tok.encode(sample, allowed_special={'<|endoftext|>'})
print('ids    :', ids)
print('decoded:', tok.decode(ids))

train_ids = tok.encode(train, allowed_special={'<|endoftext|>'})
val_ids   = tok.encode(val,   allowed_special={'<|endoftext|>'})
print(f'\ntrain tokens: {len(train_ids):,}   val tokens: {len(val_ids):,}')

GPT-2 vocab size: 50257
ids    : [3163, 73, 9613, 5158, 284, 38594, 319, 262, 13480, 286, 509, 14717, 591, 3202, 430, 220, 50256]
decoded: Arjuna spoke to Krishna on the battlefield of Kurukshetra <|endoftext|>

train tokens: 362,720   val tokens: 39,654


In [5]:
import numpy as np
from pathlib import Path

out = Path('data/processed')
np.array(train_ids, dtype=np.uint16).tofile(out / 'train.bin')
np.array(val_ids,   dtype=np.uint16).tofile(out / 'val.bin')
print('wrote train.bin / val.bin')

wrote train.bin / val.bin


## Sliding-window dataset

For a context length `T`, each training sample is:
- `x = ids[i : i+T]`
- `y = ids[i+1 : i+T+1]`  (shifted by one — next-token prediction)

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, token_ids, context_len=128, stride=128):
        self.x, self.y = [], []
        for i in range(0, len(token_ids) - context_len, stride):
            self.x.append(torch.tensor(token_ids[i:i+context_len],   dtype=torch.long))
            self.y.append(torch.tensor(token_ids[i+1:i+1+context_len], dtype=torch.long))

    def __len__(self):  return len(self.x)
    def __getitem__(self, idx): return self.x[idx], self.y[idx]

def make_loader(token_ids, batch_size=8, context_len=128, stride=128, shuffle=True):
    ds = GPTDatasetV1(token_ids, context_len=context_len, stride=stride)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=True)

loader = make_loader(train_ids, batch_size=4, context_len=64, stride=64)
xb, yb = next(iter(loader))
print('x:', xb.shape, ' y:', yb.shape)
print('x[0,:20]:', xb[0, :20].tolist())
print('y[0,:20]:', yb[0, :20].tolist())

x: torch.Size([4, 64])  y: torch.Size([4, 64])
x[0,:20]: [4202, 11, 4073, 257, 4334, 866, 48681, 286, 14966, 284, 2121, 2402, 943, 73, 9613, 13, 943, 73, 9613, 11]
y[0,:20]: [11, 4073, 257, 4334, 866, 48681, 286, 14966, 284, 2121, 2402, 943, 73, 9613, 13, 943, 73, 9613, 11, 2158]


Next: `05_attention.ipynb` — implement self-attention from scratch and stack it into causal multi-head attention.